In [1]:
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
from transformers import GPT2LMHeadModel
from transformers import GPT2TokenizerFast

model = GPT2LMHeadModel.from_pretrained("gpt2")
for name, p in model.named_parameters():
    print(name, tuple(p.shape))

/Users/cole/Desktop/projects/gpt2-audit/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1122 13:28:06.517000 59755 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10752.58it/s]

transformer.wte.weight (50257, 768)
transformer.wpe.weight (1024, 768)
transformer.h.0.ln_1.weight (768,)
transformer.h.0.ln_1.bias (768,)
transformer.h.0.attn.c_attn.weight (768, 2304)
transformer.h.0.attn.c_attn.bias (2304,)
transformer.h.0.attn.c_proj.weight (768, 768)
transformer.h.0.attn.c_proj.bias (768,)
transformer.h.0.ln_2.weight (768,)
transformer.h.0.ln_2.bias (768,)
transformer.h.0.mlp.c_fc.weight (768, 3072)
transformer.h.0.mlp.c_fc.bias (3072,)
transformer.h.0.mlp.c_proj.weight (3072, 768)
transformer.h.0.mlp.c_proj.bias (768,)
transformer.h.1.ln_1.weight (768,)
transformer.h.1.ln_1.bias (768,)
transformer.h.1.attn.c_attn.weight (768, 2304)
transformer.h.1.attn.c_attn.bias (2304,)
transformer.h.1.attn.c_proj.weight (768, 768)
transformer.h.1.attn.c_proj.bias (768,)
transformer.h.1.ln_2.weight (768,)
transformer.h.1.ln_2.bias (768,)
transformer.h.1.mlp.c_fc.weight (768, 3072)
transformer.h.1.mlp.c_fc.bias (3072,)
transformer.h.1.mlp.c_proj.weight (3072, 768)
transformer.h.

In [ ]:
from audit.model import embed, attn, layer_norm, mlp

B, T = 6, 24

x = torch.randint(50257, (B, T))
i = torch.arange(T)

embed_out = embed(x)
ln_1_out = layer_norm(embed_out, 1, 0)
attn_out = attn(ln_1_out, 0)
ln_2_out = layer_norm(attn_out, 2, 0)

x = ln_2_out

fc_weight = model.transformer.h[0].mlp.c_fc.weight
fc_bias = model.transformer.h[0].mlp.c_fc.bias
proj_weight = model.transformer.h[0].mlp.c_proj.weight
proj_bias = model.transformer.h[0].mlp.c_proj.bias

fc_out = x @ fc_weight + fc_bias

print((fc_out < 0).sum())
F.gelu(fc_out, approximate='tanh')
print((fc_out < 0).sum())
mlp_out = fc_out @ proj_weight + proj_bias

expected_mlp_out = model.transformer.h[0].mlp(x)

torch.allclose(expected_mlp_out, mlp_out, atol=1e-4)   

tensor(377411)
tensor(377409)


True

In [28]:
(torch.ones(6, 24, 768) @ torch.ones(768, 768)).shape

torch.Size([6, 24, 768])

torch.Size([6, 24, 768])